%md
### **Exploring data**

In [0]:
sales = spark.read.table("default.sales")
stores = spark.read.table("default.stores")
products = spark.read.table("default.products")
calendar = spark.read.table("default.calendar")
inventory = spark.read.table("default.inventory")

In [0]:
sales.printSchema()
print('-------------------------------------')
inventory.printSchema()
print('-------------------------------------')
calendar.printSchema()
print('-------------------------------------')
products.printSchema()
print('-------------------------------------')
stores.printSchema()

root
 |-- Sale_ID: long (nullable = true)
 |-- Date: date (nullable = true)
 |-- Store_ID: long (nullable = true)
 |-- Product_ID: long (nullable = true)
 |-- Units: long (nullable = true)

-------------------------------------
root
 |-- Store_ID: long (nullable = true)
 |-- Product_ID: long (nullable = true)
 |-- Stock_On_Hand: long (nullable = true)

-------------------------------------
root
 |-- Date: date (nullable = true)

-------------------------------------
root
 |-- Product_ID: long (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Cost: string (nullable = true)
 |-- Product_Price: string (nullable = true)

-------------------------------------
root
 |-- Store_ID: long (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Store_City: string (nullable = true)
 |-- Store_Location: string (nullable = true)
 |-- Store_Open_Date: date (nullable = true)



In [0]:
sales.createOrReplaceTempView("sales")
products.createOrReplaceTempView("products")
calendar.createOrReplaceTempView("calendar")
inventory.createOrReplaceTempView("inventory")
stores.createOrReplaceTempView("stores")

In [0]:
spark.sql(""" 
          select * from products
          """).show()

+----------+--------------------+-----------------+------------+-------------+
|Product_ID|        Product_Name| Product_Category|Product_Cost|Product_Price|
+----------+--------------------+-----------------+------------+-------------+
|         1|       Action Figure|             Toys|      $9.99 |      $15.99 |
|         2|      Animal Figures|             Toys|      $9.99 |      $12.99 |
|         3|     Barrel O' Slime|     Art & Crafts|      $1.99 |       $3.99 |
|         4|    Chutes & Ladders|            Games|      $9.99 |      $12.99 |
|         5|    Classic Dominoes|            Games|      $7.99 |       $9.99 |
|         6|           Colorbuds|      Electronics|      $6.99 |      $14.99 |
|         7|            Dart Gun|Sports & Outdoors|     $11.99 |      $15.99 |
|         8|       Deck Of Cards|            Games|      $3.99 |       $6.99 |
|         9|            Dino Egg|             Toys|      $9.99 |      $10.99 |
|        10|    Dinosaur Figures|             Toys| 

In [0]:
products_cleaned = spark.sql(""" 
        SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products

          """).show()

+----------+--------------------+-----------------+------------+-------------+
|Product_ID|        Product_Name| Product_Category|Product_Cost|Product_Price|
+----------+--------------------+-----------------+------------+-------------+
|         1|       Action Figure|             Toys|        9.99|        15.99|
|         2|      Animal Figures|             Toys|        9.99|        12.99|
|         3|     Barrel O' Slime|     Art & Crafts|        1.99|         3.99|
|         4|    Chutes & Ladders|            Games|        9.99|        12.99|
|         5|    Classic Dominoes|            Games|        7.99|         9.99|
|         6|           Colorbuds|      Electronics|        6.99|        14.99|
|         7|            Dart Gun|Sports & Outdoors|       11.99|        15.99|
|         8|       Deck Of Cards|            Games|        3.99|         6.99|
|         9|            Dino Egg|             Toys|        9.99|        10.99|
|        10|    Dinosaur Figures|             Toys| 

### **Data Analysis with SQL**

In [0]:
showStable = spark.sql("""select*from sales""")
showStable.show(5)

+-------+----------+--------+----------+-----+
|Sale_ID|      Date|Store_ID|Product_ID|Units|
+-------+----------+--------+----------+-----+
|      1|2022-01-01|      24|         4|    1|
|      2|2022-01-01|      28|         1|    1|
|      3|2022-01-01|       6|         8|    1|
|      4|2022-01-01|      48|         7|    1|
|      5|2022-01-01|      44|        18|    1|
+-------+----------+--------+----------+-----+
only showing top 5 rows


In [0]:
showPtable = spark.sql("""select*from products""")
showPtable.show(5)

+----------+----------------+----------------+------------+-------------+
|Product_ID|    Product_Name|Product_Category|Product_Cost|Product_Price|
+----------+----------------+----------------+------------+-------------+
|         1|   Action Figure|            Toys|      $9.99 |      $15.99 |
|         2|  Animal Figures|            Toys|      $9.99 |      $12.99 |
|         3| Barrel O' Slime|    Art & Crafts|      $1.99 |       $3.99 |
|         4|Chutes & Ladders|           Games|      $9.99 |      $12.99 |
|         5|Classic Dominoes|           Games|      $7.99 |       $9.99 |
+----------+----------------+----------------+------------+-------------+
only showing top 5 rows


In [0]:
cityByPriceQteRevenue = spark.sql("""
    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)
        
    SELECT 
        t.store_city as city, 
        -- TRUNCATE(SUM(p.Product_Price),2) as Price,
        FLOOR(SUM(p.Product_Price)*100)/100 as Price,
        count(s.Sale_ID) as count_sales,
        SUM(s.Units) as sum_qte,
        FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
        floor((sum(p.Product_Price * s.Units - p.Product_Cost * s.Units))*100)/100 AS gross_profit

    FROM sales s LEFT JOIN stores t ON s.Store_ID = t.Store_ID 
    LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID
    GROUP BY t.store_city
    ORDER BY gross_profit DESC
""")

cityByPriceQteRevenue.show()

+----------------+----------+-----------+-------+-----------+------------+
|            city|     Price|count_sales|sum_qte|sum_revenue|gross_profit|
+----------------+----------+-----------+-------+-----------+------------+
|Cuidad de Mexico|1251716.74|      90725| 125599|  1649492.0|    465558.0|
|     Guadalajara|1047609.19|      74380|  96454| 1322099.45|    368930.0|
|       Monterrey| 962036.76|      69323|  93730| 1261845.69|    346729.0|
|      Hermosillo| 690696.64|      49835|  66816|  903388.83|    263608.0|
|      Guanajuato| 689515.79|      49220|  64317|  869055.82|    235047.0|
|          Puebla| 652895.91|      47408|  61171|  808710.28|    229694.0|
|        Mexicali| 463489.44|      33855|  44415|  586175.84|    175048.0|
|          Xalapa| 483698.92|      33807|  44223|  610119.76|    163720.0|
|        Saltillo| 465805.09|      33090|  42903|  579514.96|    163248.0|
|          Toluca|  502953.9|      36309|  48632|  633521.67|    162702.0|
|       Chihuahua| 412614

In [0]:
# max profit by city
spark.sql("""
          
with cityByPriceQteRevenue as (

    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)
        
    SELECT 
        t.store_city as city, 
        FLOOR(SUM(p.Product_Price)*100)/100 as Price,
        count(s.Sale_ID) as count_sales,
        SUM(s.Units) as sum_qte,
        FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
        floor((sum(p.Product_Price * s.Units - p.Product_Cost * s.Units))*100)/100 AS gross_profit

    FROM sales s LEFT JOIN stores t ON s.Store_ID = t.Store_ID 
    LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID
    GROUP BY t.store_city
    ORDER BY gross_profit DESC)

select city, gross_profit as max_profit
from cityByPriceQteRevenue
where gross_profit = (select max(gross_profit) from cityByPriceQteRevenue)


""").show()

+----------------+----------+
|            city|max_profit|
+----------------+----------+
|Cuidad de Mexico|  465558.0|
+----------------+----------+



In [0]:
avg_profit = spark.sql("""          
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)

        select floor(avg(gross_profit)*100)/100 as avg_profit 
        from                
            (SELECT 
                t.Store_city as city,
                floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
            from sales s
            left join cleaned_products p on s.Product_ID = p.Product_ID
            left join stores t on s.Store_ID = t.Store_ID
            group by city)
""").show()


+----------+
|avg_profit|
+----------+
| 138414.79|
+----------+



In [0]:
# profit gte avg of profit for each city
cityProfit_Gte_AvgProfit = spark.sql("""
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)

    select city, gross_profit
    from
        (SELECT 
            t.Store_city as city,
            floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
        from sales s
        left join cleaned_products p on s.Product_ID = p.Product_ID
        left join stores t on s.Store_ID = t.Store_ID
        group by city) t

    where gross_profit > (select avg(gross_profit)
                          from (
                                SELECT  
                                floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
                                from sales s
                                left join cleaned_products p on s.Product_ID = p.Product_ID
                                left join stores t on s.Store_ID = t.Store_ID
                                group by t.Store_city)
                                )
    
    order by gross_profit desc
    
""")
cityProfit_Gte_AvgProfit.show()

+----------------+------------+
|            city|gross_profit|
+----------------+------------+
|Cuidad de Mexico|    465558.0|
|     Guadalajara|    368930.0|
|       Monterrey|    346729.0|
|      Hermosillo|    263608.0|
|      Guanajuato|    235047.0|
|          Puebla|    229694.0|
|        Mexicali|    175048.0|
|          Xalapa|    163720.0|
|        Saltillo|    163248.0|
|          Toluca|    162702.0|
|       Chihuahua|    146868.0|
|        Campeche|    146339.0|
+----------------+------------+



-------------------------------------------
-------------------------------------------

In [0]:
# profit for each months in 2022
spark.sql("""
    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
    FROM products)

    SELECT 
        YEAR(s.Date) as year, 
        MONTH(s.Date) as month, 
        FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
        floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
    FROM sales s 
    LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID 
    LEFT JOIN calendar c ON s.Date = c.Date
    --where YEAR(s.Date) = 2022
    GROUP BY YEAR(s.Date), MONTH(s.Date)
    ORDER BY MONTH(s.Date) ASC

    """).show()                     

+----+-----+-----------+------------+
|year|month|sum_revenue|gross_profit|
+----+-----+-----------+------------+
|2022|    1|   542554.9|    167126.0|
|2023|    1|  747196.21|    205074.0|
|2023|    2|  722632.18|    189314.0|
|2022|    2|  541351.64|    161861.0|
|2023|    3|  883515.63|    231909.0|
|2022|    3|  589485.18|    173992.0|
|2022|    4|  681072.97|    190099.0|
|2023|    4|  827691.06|    215096.0|
|2023|    5|  825319.48|    210347.0|
|2022|    5|  672369.89|    186894.0|
|2023|    6|  808299.24|    207212.0|
|2022|    6|  661980.21|    189815.0|
|2022|    7|  556034.22|    176922.0|
|2023|    7|  828348.85|    209807.0|
|2022|    8|  489422.72|    158931.0|
|2023|    8|  660877.06|    175038.0|
|2022|    9|  585844.03|    166397.0|
|2023|    9|  658194.47|    180445.0|
|2022|   10|  623874.38|    178799.0|
|2022|   11|  661304.14|    192873.0|
+----+-----+-----------+------------+
only showing top 20 rows


In [0]:
# profit for each months in 2022 and 2023 
spark.sql("""
    with cum_profit_months_2022_2023 as (
        
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)

        SELECT 
            YEAR(s.Date) as year, 
            MONTH(s.Date) as month, 
            FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
            floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
        FROM sales s 
        LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID 
        LEFT JOIN calendar c ON s.Date = c.Date
        --where YEAR(s.Date) = 2022
        GROUP BY YEAR(s.Date), MONTH(s.Date)
        ORDER BY MONTH(s.Date) ASC
        )

select year, month , gross_profit, sum(gross_profit) over(order by month) as cumsum_profit
from cum_profit_months_2022_2023
where year = 2022 and month between 1 and 9  

    """).show()                     

+----+-----+------------+-------------+
|year|month|gross_profit|cumsum_profit|
+----+-----+------------+-------------+
|2022|    1|    167126.0|     167126.0|
|2022|    2|    161861.0|     328987.0|
|2022|    3|    173992.0|     502979.0|
|2022|    4|    190099.0|     693078.0|
|2022|    5|    186894.0|     879972.0|
|2022|    6|    189815.0|    1069787.0|
|2022|    7|    176922.0|    1246709.0|
|2022|    8|    158931.0|    1405640.0|
|2022|    9|    166397.0|    1572037.0|
+----+-----+------------+-------------+



In [0]:
# profit for each months in 2023
spark.sql("""
    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
    FROM products)
    SELECT 
        YEAR(s.Date) as year, 
        MONTH(s.Date) as month, 
        FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
        floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit

    FROM sales s 
    LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID 
    LEFT JOIN calendar c ON s.Date = c.Date
    where YEAR(s.Date) = 2023
    GROUP BY YEAR(s.Date), MONTH(s.Date)
    ORDER BY MONTH(s.Date) ASC

    """).show()                     

+----+-----+-----------+------------+
|year|month|sum_revenue|gross_profit|
+----+-----+-----------+------------+
|2023|    1|  747196.21|    205074.0|
|2023|    2|  722632.18|    189314.0|
|2023|    3|  883515.63|    231909.0|
|2023|    4|  827691.06|    215096.0|
|2023|    5|  825319.48|    210347.0|
|2023|    6|  808299.24|    207212.0|
|2023|    7|  828348.85|    209807.0|
|2023|    8|  660877.06|    175038.0|
|2023|    9|  658194.47|    180445.0|
+----+-----+-----------+------------+



In [0]:
# profit for each months in 2022 and 2023 
spark.sql("""
    with cum_profit_months_2022_2023 as (
        
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)

        SELECT 
            YEAR(s.Date) as year, 
            MONTH(s.Date) as month, 
            FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
            floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
        FROM sales s 
        LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID 
        LEFT JOIN calendar c ON s.Date = c.Date
        --where YEAR(s.Date) = 2022
        GROUP BY YEAR(s.Date), MONTH(s.Date)
        ORDER BY MONTH(s.Date) ASC
        )

select year, month , gross_profit, sum(gross_profit) over(order by month) as cumsum_profit
from cum_profit_months_2022_2023
where year = 2023   

    """).show()                     

+----+-----+------------+-------------+
|year|month|gross_profit|cumsum_profit|
+----+-----+------------+-------------+
|2023|    1|    205074.0|     205074.0|
|2023|    2|    189314.0|     394388.0|
|2023|    3|    231909.0|     626297.0|
|2023|    4|    215096.0|     841393.0|
|2023|    5|    210347.0|    1051740.0|
|2023|    6|    207212.0|    1258952.0|
|2023|    7|    209807.0|    1468759.0|
|2023|    8|    175038.0|    1643797.0|
|2023|    9|    180445.0|    1824242.0|
+----+-----+------------+-------------+



In [0]:
# profit for each months in 2022 and 2023

spark.sql("""
    
select 
    year, 
    month , 
    gross_profit, 
    sum(gross_profit) over(order by month) as cumsum_profit,
    sum(gross_profit) over(partition by year order by month) as cumsum_profit_yearly
from
    (
    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
    FROM products)

    SELECT 
        YEAR(s.Date) as year, 
        MONTH(s.Date) as month, 
        FLOOR(SUM(p.Product_Price * s.Units)*100)/100 AS sum_revenue,
        floor(sum(p.Product_Price*s.Units - p.Product_Cost*s.Units)*100)/100 as gross_profit
    FROM sales s 
    LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID 
    LEFT JOIN calendar c ON s.Date = c.Date
    where month(s.Date) BETWEEN 1 and 9
    GROUP BY YEAR(s.Date), MONTH(s.Date)
    ORDER BY MONTH(s.Date) ASC)t
    
order by year
    """).show()                     

+----+-----+------------+-------------+--------------------+
|year|month|gross_profit|cumsum_profit|cumsum_profit_yearly|
+----+-----+------------+-------------+--------------------+
|2022|    1|    167126.0|     372200.0|            167126.0|
|2022|    2|    161861.0|     723375.0|            328987.0|
|2022|    3|    173992.0|    1129276.0|            502979.0|
|2022|    4|    190099.0|    1534471.0|            693078.0|
|2022|    5|    186894.0|    1931712.0|            879972.0|
|2022|    6|    189815.0|    2328739.0|           1069787.0|
|2022|    7|    176922.0|    2715468.0|           1246709.0|
|2022|    8|    158931.0|    3049437.0|           1405640.0|
|2022|    9|    166397.0|    3396279.0|           1572037.0|
|2023|    1|    205074.0|     372200.0|            205074.0|
|2023|    2|    189314.0|     723375.0|            394388.0|
|2023|    3|    231909.0|    1129276.0|            626297.0|
|2023|    4|    215096.0|    1534471.0|            841393.0|
|2023|    5|    210347.0

In [0]:
# profit for each year
spark.sql("""
    with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
    FROM products)

    select year(s.Date) ,floor(sum((p.Product_Price * s.Units)-(p.Product_Cost * s.Units))*100)/100 as gross_profit        
    from sales s 
    left join cleaned_products p ON s.Product_ID=p.Product_ID
    left join calendar c ON s.Date = c.Date
    where month(s.Date) between 1 and 9 
    group by year(s.Date)        
          
          """).show()

+------------+------------+
|year(s.Date)|gross_profit|
+------------+------------+
|        2023|   1824242.0|
|        2022|   1572037.0|
+------------+------------+



In [0]:
count_Of_catgs_Sold = spark.sql("""
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)

        Select p.Product_Category, count(*) as count_sold_product
        from sales s left join cleaned_products p on S.Product_ID = p.Product_ID
        group by P.Product_Category
        order by count_sold_product desc
          
        """)
count_Of_catgs_Sold.show()

+-----------------+------------------+
| Product_Category|count_sold_product|
+-----------------+------------------+
|             Toys|            221227|
|     Art & Crafts|            220673|
|            Games|            157006|
|Sports & Outdoors|            131331|
|      Electronics|             99025|
+-----------------+------------------+



In [0]:
# Profit of categories sold
catgSold_rev = spark.sql("""  
        with cleaned_products as (SELECT Product_ID, Product_Name, Product_Category, 
            CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost, 
            CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
        FROM products)      
        select p.Product_Category, count(*) as count_sold_product, 
                floor(sum(p.Product_Price * s.Units)*100)/100 as revenue,
                floor(sum(p.Product_Cost * s.Units)*100)/100 as cost,
                floor(sum(p.Product_Price * s.Units - p.Product_Cost * s.Units)*100)/100 as gross_profit
        from sales s left join cleaned_products p on s.Product_ID = p.Product_ID
        group by p.Product_Category
        order by gross_profit desc

""")
catgSold_rev.show()

+-----------------+------------------+----------+----------+------------+
| Product_Category|count_sold_product|   revenue|      cost|gross_profit|
+-----------------+------------------+----------+----------+------------+
|             Toys|            221227| 5093241.0| 4013714.0|   1079527.0|
|      Electronics|             99025|2246771.25|1245334.24|   1001437.0|
|     Art & Crafts|            220673|2705364.26|1952010.25|    753354.0|
|            Games|            157006|2226836.27|1552843.26|    673993.0|
|Sports & Outdoors|            131331|2172359.56|1666641.56|    505718.0|
+-----------------+------------------+----------+----------+------------+



In [0]:
catgSold_rev = spark.sql("""  

WITH cleaned_products AS (
    SELECT 
        Product_ID,
        Product_Name,
        Product_Category,
        CAST(regexp_replace(Product_Cost, '[\\$,]', '') AS DOUBLE) AS Product_Cost,
        CAST(regexp_replace(Product_Price, '[\\$,]', '') AS DOUBLE) AS Product_Price
    FROM products
)

SELECT 
    p.Product_Name,
    COUNT(*) AS count_sold_product,
    FLOOR(SUM(p.Product_Price * s.Units) * 100) / 100 AS revenue,
    FLOOR(SUM(p.Product_Cost * s.Units) * 100) / 100 AS cost,
    FLOOR(AVG(p.Product_Price * s.Units) * 100) / 100 AS avg_rev,
    FLOOR(SUM((p.Product_Price - p.Product_Cost) * s.Units) * 100) / 100 AS profit

FROM sales s LEFT JOIN cleaned_products p ON s.Product_ID = p.Product_ID

ORDER BY profit DESC

""")

catgSold_rev.show()


+------------------+------------------+----------+----------+-------+---------+
|      Product_Name|count_sold_product|   revenue|      cost|avg_rev|   profit|
+------------------+------------------+----------+----------+-------+---------+
|         Colorbuds|             72988|1564476.31| 729532.31|  21.43| 834944.0|
|     Action Figure|             48497| 926748.41| 579000.41|   19.1| 347748.0|
|       Lego Bricks|             48030|2388882.63|2090197.62|  49.73| 298685.0|
|     Deck Of Cards|             68083| 587397.65| 335295.65|   8.62| 252102.0|
|     Glass Marbles|             24507| 412322.81| 224732.81|  16.82| 187590.0|
|   Barrel O' Slime|             54078| 365735.36| 182409.36|   6.76| 183326.0|
|   Kids Makeup Kit|             21648| 488415.66| 341817.66|  22.56|146597.99|
|          Nerf Gun|             23709| 530594.56| 397879.56|  22.37|132714.99|
|          Dart Gun|             26203| 505092.11| 378740.11|  19.27| 126352.0|
|     Etch A Sketch|             11205| 